<a href="https://colab.research.google.com/github/uu-sml/wasp-assigninmen-af-classification/blob/main/assignment_ecg_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WASP Course: Artificial Intelligence and Machine Learning

Lecturer: Dave Zachariah

Assignment responsible: Jingwei Hu, Steven Wang, David Vävinggren

# Student and Group Information

Fill this out for the submission of the assignment (you submit this notebook with your solution)

- **Student names:** <font color='red'>Fill in</font>

- **Team ID:** <font color='red'>Fill in (see instructions below)</font>

> ### Naming Convention
>
> Your Team ID must match your Studium group number. Use the format group_`<N>` where `<N>` is your group number in Studium. For example, if you are **Studium group 11**, your Team ID is **`group_11`**.
>
> Please use this exact same Team ID everywhere: here, when you register on the leaderboard (Coding Task 7), and in every submission.
>
> Your three submission notes should follow the format:
>
> ```
> group_<N>_sub_1
> group_<N>_sub_2
> group_<N>_sub_3
> ```
>
> e.g. for Studium group 11: `group_11_sub_1`, `group_11_sub_2`, `group_11_sub_3`.
>
> If your Team ID does not match your Studium group number, we cannot map your submission to your group. Double-check that you have followed these instructions before you register and submit.


---
# Module 3 - Assignment Overview: ECG classification

The [electrocardiogram (ECG)](https://www.mayoclinic.org/tests-procedures/ekg/about/pac-20384983) records the electrical signals in the heart. It is a common  test used to quickly detect heart problems and to monitor the heart's health. 
In this assignment you will implement and evaluate a model to classify whether the person has [atrial fibrillation (AF)](https://www.mayoclinic.org/diseases-conditions/atrial-fibrillation/symptoms-causes/syc-20350624.) or not based on measurements from the ECG exam. 


**Submission:** You submit the deliverables (see below) at https://canvas.kth.se/courses/63382/assignments

**Due Date:** August 22, 2026.

---
## Basic Tasks
Your task is to implement a classification model, train this model on training data, and evaluate its performance on validation data. We provide skeleton code for the implementation of a simple convolution neural network model.

The steps required to implement this model are presented as numbered tasks below. In total there are seven (7) coding tasks and five (5) explanation tasks. 

## Competitive setting

You have to compute the predictions for the test data (you do not have the labels for it) and submit your predictions to be evaluated to a leaderboard. These predictions will be scored and your submission will be ranked according to the F1 score and compared with your colleagues. In the end a winning team will be determined.

### Deliverables
There are two deliverables:
1. You have to submit this Jupyter notebook on the course web-page (Canvas) together with your code and explanations (where asked for it) that describe your implementation and your experimental results. The notebook should run as a standalone in google colab.
2. You have to have at least **three (3)** submissions (for instructions on how to submit, see coding task 7) where you try to improve the model architecture, the training procedure or the problem formulation. In the submission of this notebook you have to provide a short explanation of what changed between each submission and justify why you decided to make these changes.

### Grading
To pass the assignment, you must submit a complete and working implementation of a model and a well-motivated description and evaluation of it. Your model should reach an Area under the ROC curve (AUROC) on the test data of at least 0.97 and an Average Precision (AP) score of 0.95. Note that the leaderboard to is sorted by F1 score and not AUROC, hence you would want to balance all three metrics.

### GPU Acceleration
To be able to use the GPUs provided by colab in order to speed up your computations, you want to check that the `Hardware accelerator` is set to `GPU` under `Runtime > change runtime type`. Note that notebooks run by connecting to virtual machines that have maximum lifetimes that can be as much as 12 hours. Notebooks will also disconnect from VMs when left idle for too long. 

In [ ]:
import os

# helper function
def exists(path):
    return os.path.exists(path)

# Download requirements.txt directly from your repo if not present
if not exists('requirements.txt'):
    # Note: Using the raw.githubusercontent URL ensures you get the file content, not the HTML page
    repo_url = "https://raw.githubusercontent.com/uu-sml/wasp-assignmen-af-classification/main/requirements.txt"
    !curl -O {repo_url}
    print("Downloaded latest requirements.txt from repository.")

# Install the dependencies
!pip install -r requirements.txt --upgrade --no-cache-dir

In [ ]:
# Import
import torch
import torch.nn as nn
import numpy as np
from tqdm.notebook import trange, tqdm
import h5py
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

---
## The data set

The dataset is a subset of the [*CODE dataset*](https://scilifelab.figshare.com/articles/dataset/CODE_dataset/15169716): an anotated database of ECGs. The ECG exams were recorded in Brazil by the Telehealth Network of the state Minas Gerais between 2010 and 2016. The dataset and its usage for the development of deep learning methods was described in ["Automatic diagnosis of the 12-lead ECG using a deep neural network"](https://www.nature.com/articles/s41467-020-15432-4).
The full dataset is available for research upon request.


For the training dataset you have labels. 
For the test dataset you only have the ECG exams but no labels. Evaluation is done by submitting to the leaderboard.

Download the dataset from the given Google Drive link and unzip the folder containing the files. The downloaded files are in WFDB format (see [here](https://www.physionet.org/content/wfdb-python/3.4.1/) for details).

In [ ]:
# 1. Download dataset
if not exists('codesubset.tar.gz'):
    !pip install gdown
    !gdown "https://drive.google.com/uc?id=1zg2WoSbu6-5980X9hnpxDkZTwkYDbDdH" -O codesubset.tar.gz

In [ ]:
# 1. unzip the downloaded data set folder
if not exists('codesubset'):
    !tar -xf codesubset.tar.gz

Note that the extraced folder 'codesubset' contains
1. subfolders with the ECG exam traces. These have to be further preprocessed which we do in the next steps.
2. a csv file which contain the labels and other features for the training data set.


### Preprocessing

Run the cells below to  Clone the GitHub repository which we use for [data preprocessing](https://github.com/antonior92/ecg-preprocessing).

In [ ]:
# 2. clone the code files for data preprocessing
if not exists('ecg-preprocessing'):
    !git clone https://github.com/paulhausner/ecg-preprocessing.git

Let us plot an ECG sample. We can plot ECGs using the `ecg_plot` library for example by using the following code snippet where `ecg_sample` is an array of size `(number of leads * sequence length)`. Now we can view an ECG before preprocessing. Should you not have the library, you may install the library by running the command `!pip install ecg_plot`.

In [ ]:
!pip install ecg_plot

In [ ]:
import ecg_plot
runfile("ecg-preprocessing/read_ecg.py")

PATH_TO_WFDB = 'codesubset/train/TNMG100046'
ecg_sample, sample_rate, _ = read_ecg(PATH_TO_WFDB)

# ECG plot
plt.figure()
lead = ['I', 'II', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
ecg_plot.plot(ecg_sample, sample_rate=sample_rate, style='bw', row_height=8, lead_index=lead, columns=1, title='Sample ECG before pre-processing')
plt.show()


The preprocessing consist of:
- resampling all ECG traces to the sample sampling period (400 Hz). Option: ``--new_freq 400``
- zero padding if necessary such that all ECG have the same number of samples (4096). Option: ``--new_len 4096``.
- removing trends in the ECG signal. Option: ``--remove_baseline``
- remove possible power line noise. Option: ``--powerline 60``

You can run the script bellow to plot the same ECG after the preprocessing.  The script also use the  `ecg_plot` library (as you did above).  You can try also with different command line options to see how the preprocessing affects the signal that will be used by the model.

In [ ]:
%run ecg-preprocessing/plot_from_ecg.py codesubset/train/TNMG100046 --new_freq 400 --new_len 4096 --remove_baseline --powerline 60


Next we perform the preprocessing in all exams and convert them into one single h5 file (see [here](https://www.h5py.org/#:~:text=The%20h5py%20package%20is%20a,they%20were%20real%20NumPy%20arrays.) for details about the format). The resulting h5 files contains the traces as arrays with the shape `(number of traces * sequence length * number of leads)` where sequence length is 4096 and number of leads is 8. 
The files `train.h5` and `test.h5` will be saved inside the folder `codesubset/`.

In [ ]:
# 3. Generate train
if not exists('codesubset/train.h5'):
    !python ecg-preprocessing/generate_h5.py --new_freq 400 --new_len 4096 --remove_baseline --powerline 60 codesubset/train/RECORDS.txt codesubset/train.h5
# 3. Generate test
if not exists('codesubset/test.h5'):
    !python ecg-preprocessing/generate_h5.py --new_freq 400 --new_len 4096 --remove_baseline --powerline 60 codesubset/test/RECORDS.txt codesubset/test.h5

### Coding Task 1: Data Analysis

Before starting to model you have to analyse the dataset. You can be creative in your way of *getting a feeling* for the data. What you have to do is:
- plot an ECG after proprocessing saved in the hdf5 file. For this use the `ecg_plot()` example above and see below for how to access the preprocessed data in h5 format.

Some further ideas to explore are:
- check the balance of the data set,
- evaluate the distribution of age and sex of the patients,
- think about the performance that a best naive classifier would achieve, e.g. by random guessing or always predicting one class.

<br />

**How to access the data?**

You can acces the data in the h5 file in the following way
```
import h5py

PATH_TO_H5_FILE = 'codesubset/train.h5'
f = h5py.File(PATH_TO_H5_FILE, 'r')
data = f['tracings']
```
Then, `data[i]` is an numpy array of the $i$th ECG exam (including all time points and leads).


In [ ]:
PATH_TO_H5_FILE = "codesubset/train.h5"
PATH_TO_CSV_FILE = "codesubset/train.csv"
PATH_TO_RECORDS_FILE = "codesubset/train/RECORDS.txt"

with h5py.File(PATH_TO_H5_FILE, "r") as h5_file:
    tracings = h5_file["tracings"]
    n_exams, sequence_length, n_leads = tracings.shape
    ecg_sample = tracings[0]

records = pd.read_csv(PATH_TO_RECORDS_FILE, header=None)[0].astype(str)
ids_traces = records.str.split("TNMG", expand=True)[1].astype(int).to_numpy()

df = pd.read_csv(PATH_TO_CSV_FILE).set_index("id_exam").reindex(ids_traces)
labels = df["AF"].astype(int)

positive_rate = labels.mean()
majority_accuracy = max(positive_rate, 1.0 - positive_rate)

print(f"Number of ECG exams: {n_exams}")
print(f"Trace shape: {sequence_length} time points x {n_leads} leads")
print(f"AF positive examples: {labels.sum()} / {len(labels)} ({positive_rate:.1%})")
print(f"Majority-class naive accuracy: {majority_accuracy:.1%}")

display_columns = [column for column in ["age", "sex", "AF"] if column in df.columns]
display(df[display_columns].describe(include="all"))

plt.figure(figsize=(14, 8))
for lead_index, lead_name in enumerate(["I", "II", "V1", "V2", "V3", "V4", "V5", "V6"]):
    plt.plot(ecg_sample[:, lead_index] + lead_index * 4, label=lead_name)
plt.title("Example preprocessed ECG from train.h5")
plt.xlabel("Sample")
plt.ylabel("Amplitude offset by lead")
plt.legend(loc="upper right")
plt.show()


### Explanation task 1: Data Analysis

- The preprocessed data are stored as a single tensor of ECG traces with shape `(number of exams, 4096, 8)`, which makes them directly usable as model input.
- The AF label distribution should be checked before training because class imbalance can make accuracy misleading. A majority-class baseline gives a useful lower bound: a model must beat this while also improving ranking metrics and F1.
- Age and sex summaries help reveal whether the dataset is demographically skewed, although the model below uses only the ECG trace as input.
- The preprocessing resamples all recordings to a common sampling rate, pads or truncates them to the same length, removes baseline drift, and filters power-line noise. This is necessary because a neural network batch requires equal-length inputs and benefits from reducing non-cardiac signal artifacts.


---
## Model

The model class consists of two methods: 
- `__init__(self, args)`: This methods initializes the class, e.g. by using `mymodel=ModelBaseline(args)`.
- `forward(self,input_data)`: This method is called when we run `model_output=mymodel(input_data)`.

The dimension of the input data is  `(batch size * sequence length * number of leads)`. Where **batch size** is a hyperparameter, **sequence length** is the number of ECG time samples (=4096) and **number of leads** (=8).

The `ModelBaseline` (provided below) is a 2 layer model with one convolutional layers and one linear layer. Some explanations: 
- The conv layer downsamples the input traces from 4096 samples to 128 samples and increases the number of channels from 8 (=number of leads) to 32. Here we use a kernel size of 3.
- The linear layer uses the flattened output from the conv and outputs one prediction. Since we have a binary problem, a single prediction is sufficient.


In [ ]:
class ModelBaseline(nn.Module):
    def __init__(self,):
        super(ModelBaseline, self).__init__()
        self.kernel_size = 3

        # conv layer
        downsample = self._downsample(4096, 128)
        self.conv1 = nn.Conv1d(in_channels=8, 
                               out_channels=32, 
                               kernel_size=self.kernel_size, 
                               stride=downsample,
                               padding=self._padding(downsample),
                               bias=False)
        
        # linear layer
        self.lin = nn.Linear(in_features=32*128,
                             out_features=1)
        
        # ReLU
        self.relu = nn.ReLU()

    def _padding(self, downsample):
        return max(0, int(np.floor((self.kernel_size - downsample + 1) / 2)))

    def _downsample(self, seq_len_in, seq_len_out):
        return int(seq_len_in // seq_len_out)


    def forward(self, x):
        x= x.transpose(2,1)

        x = self.relu(self.conv1(x))
        x_flat= x.view(x.size(0), -1)
        x = self.lin(x_flat)

        return x

### Coding Task 2: Define your model

In the cell below you have to define your model. You can be inspired by the baseline model above but you can also define any other kind of neural network architecture.

In [ ]:
class SqueezeExcitation1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(channels, hidden, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(hidden, channels, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.gate(x)


class InceptionBlock1D(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_sizes=(9, 19, 39),
        bottleneck_channels=32,
        dropout=0.10,
    ):
        super().__init__()
        self.bottleneck = nn.Conv1d(
            in_channels,
            bottleneck_channels,
            kernel_size=1,
            bias=False,
        )
        self.branches = nn.ModuleList(
            [
                nn.Conv1d(
                    bottleneck_channels,
                    out_channels,
                    kernel_size=kernel_size,
                    padding=kernel_size // 2,
                    bias=False,
                )
                for kernel_size in kernel_sizes
            ]
        )
        self.pool_branch = nn.Sequential(
            nn.MaxPool1d(kernel_size=3, stride=1, padding=1),
            nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False),
        )
        merged_channels = out_channels * (len(kernel_sizes) + 1)
        self.norm = nn.BatchNorm1d(merged_channels)
        self.activation = nn.ReLU(inplace=True)
        self.se = SqueezeExcitation1D(merged_channels)
        self.dropout = nn.Dropout(dropout)
        self.shortcut = (
            nn.Sequential(
                nn.Conv1d(in_channels, merged_channels, kernel_size=1, bias=False),
                nn.BatchNorm1d(merged_channels),
            )
            if in_channels != merged_channels
            else nn.Identity()
        )

    def forward(self, x):
        bottleneck = self.bottleneck(x)
        branches = [branch(bottleneck) for branch in self.branches]
        branches.append(self.pool_branch(x))
        merged = self.activation(self.norm(torch.cat(branches, dim=1)))
        merged = self.dropout(self.se(merged))
        return self.activation(merged + self.shortcut(x))


class TemporalDownsample1D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm1d(channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.layer(x)


class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            InceptionBlock1D(8, 32, bottleneck_channels=16, dropout=0.08),
            TemporalDownsample1D(128),
            InceptionBlock1D(128, 32, bottleneck_channels=32, dropout=0.10),
            TemporalDownsample1D(128),
            InceptionBlock1D(128, 48, bottleneck_channels=32, dropout=0.12),
            TemporalDownsample1D(192),
            InceptionBlock1D(192, 64, bottleneck_channels=48, dropout=0.15),
        )
        self.average_pool = nn.AdaptiveAvgPool1d(1)
        self.maximum_pool = nn.AdaptiveMaxPool1d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.30),
            nn.Linear(512, 1),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.features(x)
        pooled = torch.cat(
            [self.average_pool(x).flatten(1), self.maximum_pool(x).flatten(1)],
            dim=1,
        )
        return self.classifier(pooled)


### Explanation Task 2: Final Model

- The final model is an InceptionTime-style 1D CNN that operates directly on the eight preprocessed ECG leads.
- Each inception block applies several temporal convolution kernels in parallel. Shorter kernels can capture sharp QRS morphology, while longer kernels can capture slower rhythm patterns relevant for atrial fibrillation.
- Residual shortcuts help gradients flow through the deeper network and make training more stable.
- Global average pooling keeps the classifier head small and reduces overfitting compared with flattening the full temporal feature map.
- A spectrogram or time-frequency branch is a promising future extension, but the raw-signal model is the primary implementation because it preserves precise ECG timing and morphology.


---
## Train function

The function `train(...)` is called in every epoch to train the model. The function loads the training data, makes predictions, compares predictions with true labels in the loss function and adapting the model parameters using stochastic gradient descent.

In the code cell below there is the basic structure to load data from the data loader and to log your loss. The arguments of the function are explained by the use in the `main(...)` function below.

If you are unfamiliar with PyTorch training loops, then this official [tutorial](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) might help (especially section "4. Train your Network").

### Coding Task 3: Fill training loop

Fill the code cell below such that the model is training when `train(...)` is called.

In [ ]:
def augment_ecg_batch(traces):
    augmented = traces.clone()
    batch_size = augmented.shape[0]

    scale = torch.empty((batch_size, 1, 1), device=augmented.device).uniform_(0.90, 1.10)
    augmented = augmented * scale

    lead_std = augmented.std(dim=1, keepdim=True).clamp_min(1e-6)
    noise_scale = torch.empty((batch_size, 1, 1), device=augmented.device).uniform_(0.0, 0.01)
    augmented = augmented + torch.randn_like(augmented) * lead_std * noise_scale

    shifts = torch.randint(-80, 81, (batch_size,), device=augmented.device)
    for index, shift in enumerate(shifts.tolist()):
        if shift == 0:
            continue
        shifted = torch.roll(augmented[index], shifts=shift, dims=0)
        if shift > 0:
            shifted[:shift] = 0
        else:
            shifted[shift:] = 0
        augmented[index] = shifted

    dropped_leads = torch.rand((batch_size, augmented.shape[2]), device=augmented.device) < 0.10
    augmented = augmented.masked_fill(dropped_leads.unsqueeze(1), 0.0)
    return augmented


class ModelEMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.shadow = {
            name: value.detach().clone()
            for name, value in model.state_dict().items()
        }

    @torch.no_grad()
    def update(self, model):
        for name, value in model.state_dict().items():
            if self.shadow[name].is_floating_point():
                self.shadow[name].mul_(self.decay).add_(
                    value.detach(), alpha=1.0 - self.decay
                )
            else:
                self.shadow[name].copy_(value)

    def state_dict(self):
        return {name: value.detach().clone() for name, value in self.shadow.items()}


def clone_state_dict(model):
    return {
        name: value.detach().clone()
        for name, value in model.state_dict().items()
    }


def train_loop(
    epoch,
    dataloader,
    model,
    optimizer,
    loss_function,
    device,
    scaler,
    augment=False,
    ema=None,
):
    model.train()
    total_loss = 0.0
    n_entries = 0
    amp_enabled = device.type == "cuda"
    train_pbar = tqdm(dataloader, desc=f"Training Epoch {epoch:2d}", leave=True)

    for traces, diagnoses in train_pbar:
        traces = traces.to(device, non_blocking=amp_enabled)
        diagnoses = diagnoses.to(device, non_blocking=amp_enabled)
        if augment:
            traces = augment_ecg_batch(traces)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=amp_enabled):
            logits = model(traces)
            loss = loss_function(logits, diagnoses)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        if ema is not None:
            ema.update(model)

        batch_size = len(traces)
        total_loss += loss.detach().item() * batch_size
        n_entries += batch_size
        train_pbar.set_postfix(loss=total_loss / n_entries)

    train_pbar.close()
    return total_loss / n_entries


---
## Eval function

The `eval(...)` function is similar to the `train(...)` function but is used to evaluate the model on validation data without adapting the model parameters. You can prohibit computing gradients by using a `with torch.no_grad():` statement.

Currenlty only the loss is logged here. Additionally you have to collect all your predictions and the true values in order to compute more metrics such as AUROC.

### Coding Task 4: Fill evaluation loop
Fill the code cell below such we obtain model predictions to evaluate the validation loss and collect the predictoin in order to compute other validation metrics in the `main(...)` function.

In [ ]:
def eval_loop(epoch, dataloader, model, loss_function, device):
    model.eval()
    total_loss = 0.0
    n_entries = 0
    valid_logits = []
    valid_true = []
    eval_pbar = tqdm(dataloader, desc=f"Evaluation Epoch {epoch:2d}", leave=True)

    with torch.no_grad():
        for traces_cpu, diagnoses_cpu in eval_pbar:
            traces = traces_cpu.to(device, non_blocking=device.type == "cuda")
            diagnoses = diagnoses_cpu.to(device, non_blocking=device.type == "cuda")
            logits = model(traces)
            loss = loss_function(logits, diagnoses)
            valid_logits.append(logits.detach().cpu().numpy())
            valid_true.append(diagnoses.detach().cpu().numpy())

            batch_size = len(traces)
            total_loss += loss.detach().item() * batch_size
            n_entries += batch_size
            eval_pbar.set_postfix(loss=total_loss / n_entries)

    eval_pbar.close()
    return (
        total_loss / n_entries,
        np.concatenate(valid_logits, axis=0),
        np.concatenate(valid_true, axis=0),
    )


def compute_metrics(logits, targets, threshold=0.5):
    from sklearn.metrics import (
        accuracy_score,
        average_precision_score,
        f1_score,
        roc_auc_score,
    )

    scores = 1.0 / (1.0 + np.exp(-np.asarray(logits).reshape(-1)))
    target = np.asarray(targets).reshape(-1).astype(int)
    predicted = (scores >= threshold).astype(int)
    metrics = {
        "accuracy": float(accuracy_score(target, predicted)),
        "f1": float(f1_score(target, predicted, zero_division=0)),
        "average_precision": float(average_precision_score(target, scores)),
    }
    metrics["auroc"] = (
        float(roc_auc_score(target, scores))
        if np.unique(target).size == 2
        else float("nan")
    )
    return metrics


---
## Run Training

In the code cell below there are some initial (non-optimal!) training hyperparameters. Further, we combine everything from above into training code. That means that we build the dataloaders, define the model/loss/optimizer and then train/validate the model over multiple epochs. Here, we save the model with the lowest validation loss as the best model.

### Coding Task 5: Combine everything to train/validate the model

The following tasks are necessary in the code below
- split the data into training and validation data
- define the loss function
- decide and implement validation metric(s) to evaluate and compare the model on

Optional task:
- include learning rate scheduler
- take specific care about possible data inbalance

### Coding Task 6: Run your model and adapt hyperparameters

After you combined everything in task 5, now you run the code to evaluate the model. Based on the resulting validation metrics you tune
- the training hyperparameters
- the model architecture
- the model hyperparameters.

### Explanation Task 3: Hyperparameter

- I used a lower learning rate than the baseline because the final model is deeper and has batch normalization and residual connections.
- AdamW was chosen because decoupled weight decay is a stable default for convolutional neural networks.
- The batch size is kept moderate so the model can run in Colab GPU memory while still giving stable gradient estimates.
- The positive-class weight in the loss is computed from the training split to compensate for AF class imbalance without changing the validation labels or threshold.
- The best checkpoint is selected by validation F1 at threshold `0.5`, because the leaderboard is F1-oriented. AUROC and AP are still tracked to ensure the model ranks positive examples well.


In [ ]:
# set seed
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Training hyperparameters for the InceptionTime-style model.
learning_rate = 1e-3
weight_decay = 1e-4
num_epochs = 25
batch_size = 64


In [ ]:
from torch.utils.data import TensorDataset, random_split, DataLoader

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tqdm.write("Use device: {device:}\n".format(device=device))

# =============== Build data loaders ======================================#
tqdm.write("Building data loaders...")

path_to_h5_train, path_to_csv_train, path_to_records = 'codesubset/train.h5', 'codesubset/train.csv', 'codesubset/train/RECORDS.txt'
# load traces
traces = torch.tensor(h5py.File(path_to_h5_train, 'r')['tracings'][()], dtype=torch.float32)
# load labels
ids_traces = [int(x.split('TNMG')[1]) for x in list(pd.read_csv(path_to_records, header=None)[0])] # Get order of ids in traces
df = pd.read_csv(path_to_csv_train)
df.set_index('id_exam', inplace=True)
df = df.reindex(ids_traces) # make sure the order is the same
labels = torch.tensor(np.array(df['AF']), dtype=torch.float32).reshape(-1,1)
# load dataset
dataset = TensorDataset(traces, labels)
len_dataset = len(dataset)
n_classes = len(torch.unique(labels))
# split data
valid_fraction = 0.2
n_valid = int(round(len_dataset * valid_fraction))
n_train = len_dataset - n_valid
split_generator = torch.Generator().manual_seed(seed)
dataset_train, dataset_valid = random_split(dataset, [n_train, n_valid], generator=split_generator)

# build data loaders
train_dataloader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
valid_dataloader = DataLoader(dataset_valid, batch_size=batch_size, shuffle=False)                         
tqdm.write("Done!\n")


In [ ]:
# =============== Define model ============================================#
tqdm.write("Define model...")
model = Model()
model.to(device=device)
tqdm.write("Done!\n")

# =============== Define loss function ====================================#
train_labels = labels[dataset_train.indices]
num_positive = train_labels.sum()
num_negative = len(train_labels) - num_positive
pos_weight = (num_negative / num_positive).to(device) if num_positive > 0 else torch.tensor(1.0, device=device)
loss_function = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# =============== Define optimizer ========================================#
tqdm.write("Define optimiser...")
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
tqdm.write("Done!\n")

# =============== Define lr scheduler =====================================#
# Optional: reduce the learning rate when validation loss stops improving.
lr_scheduler = None

# =============== Train model =============================================#
tqdm.write("Training...")
best_f1 = -np.inf
best_loss = np.inf
# allocation
train_loss_all, valid_loss_all = [], []
valid_metrics_all = []

# loop over epochs
for epoch in trange(1, num_epochs + 1):
    # training loop
    train_loss = train_loop(epoch, train_dataloader, model, optimizer, loss_function, device)
    # validation loop
    valid_loss, y_pred, y_true = eval_loop(epoch, valid_dataloader, model, loss_function, device)

    # collect losses
    train_loss_all.append(train_loss)
    valid_loss_all.append(valid_loss)

    # compute validation metrics for performance evaluation
    y_score = y_pred.reshape(-1)
    y_target = y_true.reshape(-1).astype(int)
    y_label = (y_score >= 0.5).astype(int)

    tp = np.sum((y_label == 1) & (y_target == 1))
    fp = np.sum((y_label == 1) & (y_target == 0))
    fn = np.sum((y_label == 0) & (y_target == 1))
    tn = np.sum((y_label == 0) & (y_target == 0))

    accuracy = (tp + tn) / len(y_target)
    f1 = 0.0 if (2 * tp + fp + fn) == 0 else (2 * tp) / (2 * tp + fp + fn)

    order = np.argsort(-y_score)
    ranked_true = y_target[order]
    positive_total = np.sum(ranked_true == 1)
    precision_at_k = np.cumsum(ranked_true == 1) / (np.arange(len(ranked_true)) + 1)
    average_precision = 0.0 if positive_total == 0 else np.sum(precision_at_k[ranked_true == 1]) / positive_total

    positive_scores = y_score[y_target == 1]
    negative_scores = y_score[y_target == 0]
    if len(positive_scores) == 0 or len(negative_scores) == 0:
        auroc = np.nan
    else:
        wins = 0.0
        for positive_score in positive_scores:
            wins += np.sum(positive_score > negative_scores)
            wins += 0.5 * np.sum(positive_score == negative_scores)
        auroc = wins / (len(positive_scores) * len(negative_scores))

    metrics = {
        "epoch": epoch,
        "accuracy": accuracy,
        "f1": f1,
        "auroc": auroc,
        "average_precision": average_precision,
    }
    valid_metrics_all.append(metrics)

    # save best model: leaderboard ranking is F1-oriented, so checkpoint by validation F1.
    if (f1 > best_f1) or (np.isclose(f1, best_f1) and valid_loss < best_loss):
        torch.save({"model": model.state_dict()}, "model.pth")
        best_f1 = f1
        best_loss = valid_loss
        model_save_state = "Best model -> saved"
    else:
        model_save_state = ""

    # Print message
    tqdm.write(
        "Epoch {epoch:2d}: \t"
        "Train Loss {train_loss:.6f} \t"
        "Valid Loss {valid_loss:.6f} \t"
        "F1 {f1:.4f} \t"
        "AUROC {auroc:.4f} \t"
        "AP {average_precision:.4f} \t"
        "{model_save}".format(
            epoch=epoch,
            train_loss=train_loss,
            valid_loss=valid_loss,
            f1=f1,
            auroc=auroc,
            average_precision=average_precision,
            model_save=model_save_state,
        )
    )

    # Update learning rate with lr-scheduler
    if lr_scheduler:
        lr_scheduler.step()

plt.figure(figsize=(8, 5))
plt.plot(train_loss_all, label="train loss")
plt.plot(valid_loss_all, label="validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Learning curves")
plt.show()

pd.DataFrame(valid_metrics_all)


---
## Model Testing

Since we saved our best model, we can now load the trained model and make predictions on the test data set. We save the predictions in a csv file which will be uploaded as part of the deliverables. Note that we take a `Sigmoid()` function on the model prediction in order to obtain soft predictions (probabilities) instead of hard predictions (0s or 1s).

### Coding Task 7: Make prediction for test data

Here you do not really need to code but you have to:
- replace the baseline model with your model. If you do not use colab then change the path to the model location to load the trained model)
- run the script. The predictions are saved in the variable `soft_pred`.
- upload your predictions to the leaderboard online (see instruction details below). 

In [ ]:
# build the dataloader once and re-use when running the cell below possibly multiple times.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =============== Build data loaders ==========================================#
tqdm.write("Building data loaders...")
# load data
path_to_h5_test, path_to_csv_test = 'codesubset/test.h5', 'codesubset/test.csv'
traces = torch.tensor(h5py.File(path_to_h5_test, 'r')['tracings'][()], dtype=torch.float32)
dataset = TensorDataset(traces)
len_dataset = len(dataset)
# build data loaders
test_dataloader = DataLoader(dataset, batch_size=32, shuffle=False)
tqdm.write("Done!\n")

In [ ]:
# =============== Define model ================================================#
tqdm.write("Define model...")
"""
TASK: Replace the baseline model with your model; Insert your code here
"""
model = Model()

# load stored model parameters
ckpt = torch.load('model.pth', map_location=lambda storage, loc: storage)
model.load_state_dict(ckpt['model'])
# put model on device
model.to(device=device)
tqdm.write("Done!\n")

# =============== Evaluate model ==============================================#
model.eval()
# allocation
test_pred = torch.zeros(len_dataset,1)
# progress bar def
test_pbar = tqdm(test_dataloader, desc="Testing")
# evaluation loop
end=0
for traces in test_pbar:
    # data to device
    traces = traces[0].to(device)
    start = end
    with torch.no_grad():
        # Forward pass
        model_output = model(traces)

        # store output
        end = min(start + len(model_output), test_pred.shape[0])
        test_pred[start:end] = torch.nn.Sigmoid()(model_output).detach().cpu()

test_pbar.close()

# =============== Save predictions ============================================#
soft_pred = np.stack((1-test_pred.numpy(), test_pred.numpy()),axis=1).squeeze()

To upload your predictions to the leaderboard, use the following code. There are the following steps to follow:
1. Download the GitHub repository for the leaderboard submission system.
2. Register your team with a **team id** and **password**. The password ensures that only your team can upload to your team id. Only run the registration once.
3. Upload you predictions as a new submission. There are some things to obey here:
    - For each submission you have to attach a **note** so we know which submission you refer to in your explanation.
    - You can only get one prediction evaluated per day and you get the score the following day. If you do multiple submissions on the same day, the initial submission will be overwritten and thus only the final submission will be evaluated.
    - Only a maximum of ***FIVE*** submissions will be evaluated. So make them count! (If you update a submission before it is evaluated it doesn't count)
    - The evaluation score is published with you team_id and note at http://hyperion.it.uu.se:5050/leaderboard

> ### Naming Convention (see also the top of the notebook)
>
> - Your `team_id` should be `group_<N>`, where `<N>` is your Studium group number (e.g. Studium group 11 has `team_id = 'group_11'`). This must be identical to the Team ID you filled in at the top of this notebook.
> - Each submission `note` must be `group_<N>_sub_<k>`, where `<k>` is the submission number `1`, `2` or `3` (e.g. `group_11_sub_1`, `group_11_sub_2`, `group_11_sub_3`).
>
> **Do not use free-form names** (such as "submission A" or "model B"). Using this convention helps us a lot when marking your reports.


In [ ]:
# 1. Download repository for leaderboard submission system
if not exists('leaderboard'):
    !git clone https://gist.github.com/3ff6c4c867331c0bf334301842d753c7.git leaderboard

In [ ]:
# 2. Registration of your team
host = "http://hyperion.it.uu.se:5050/"
runfile("leaderboard/leaderboard_helpers.py")

"""
TASK: Set your team_id and a password.

This must match the Team ID at the top of the notebook.
Do not change team_id/password after you have registered your team.
(max 20 chars)
"""
team_id = '' # e.g. 'group_11'
password = '' # fill in a string

# run the registration
r = register_team(team_id, password)
if (r.status_code == 201):
    print("Team registered successfully! Good luck")
elif not (r.status_code == 200):
    raise Exception("You can not change your password once created. If you need help, please contact the teachers")


In [ ]:
# 3. Upload the prediction as submission

"""
TASK: Set the note for this submission.
"""

note = '' # e.g. 'group_11_sub_1'

# Submit the predictions to the leaderboard. Note, this also saves your submissions in your colab folder
r = submit(team_id, password, soft_pred.tolist(), note)
if r.status_code == 201:
    print("Submission successful!")
elif r.status_code == 200:
    print("Submission updated!")


### Explanation Task 4: Submissions

The final submission table should be filled after running the notebook in Colab/cloud and submitting three leaderboard attempts. The intended progression is:

- `group_<N>_sub_1`: baseline model from the assignment notebook.
- `group_<N>_sub_2`: InceptionTime-style raw ECG model from this notebook.
- `group_<N>_sub_3`: tuned InceptionTime-style model after adjusting epochs, dropout, or learning rate based on validation results.

Your team id: **<font color='red'>Fill in</font>**

| Submission note      | Accuracy | F1 | AUC | AP | Submission description |
| -------------------- | -------- | -- | --  | -- | ---------------------- |
| group_N_sub_1        | Fill in  | Fill in | Fill in | Fill in | Baseline CNN from the assignment |
| group_N_sub_2        | Fill in  | Fill in | Fill in | Fill in | InceptionTime-style 1D CNN on raw ECG traces |
| group_N_sub_3        | Fill in  | Fill in | Fill in | Fill in | Tuned InceptionTime-style model |


### Explanation Task 5: Reflection on Metrics

- Accuracy measures the fraction of correct hard predictions, but it can be misleading when AF is rare because a model can achieve high accuracy by predicting the majority class.
- F1 balances precision and recall for the positive AF class. It is important for the leaderboard because it rewards models that find AF cases without producing too many false positives.
- AUROC measures how well the model ranks positive examples above negative examples across all possible thresholds. It is threshold-independent, so it can look strong even if the fixed threshold used for F1 is poorly calibrated.
- Average precision summarizes precision-recall behavior and is especially informative for imbalanced binary classification.
- Optimizing only AUROC can produce a model with good ranking but poor performance at the chosen decision threshold. For this reason, the notebook tracks AUROC, AP, and F1 together.
